In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('../../data/raw/wb_income_classification.csv', encoding='latin-1', sep=';')

Step 1: Use exploratory function

In [5]:
def analyze_dataframe(df):
    """
    Comprehensive DataFrame analysis showing missing values, unique values, and value counts
    """
    
    print("=" * 80)
    print("DATAFRAME ANALYSIS")
    print("=" * 80)
    
    # Section 1: Basic Info
    print("\n1. BASIC DATAFRAME INFORMATION")
    print("-" * 40)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Total cells: {df.size}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Section 2: Missing Values Analysis
    print("\n2. MISSING VALUES BY COLUMN")
    print("-" * 40)
    
    # Create a DataFrame with missing value info
    missing_info = pd.DataFrame({
        'Column': df.columns,
        'Data Type': [df[col].dtype for col in df.columns],
        'Non-Null Count': [df[col].count() for col in df.columns],
        'Missing Count': [df[col].isnull().sum() for col in df.columns],
        'Missing %': [round(df[col].isnull().sum() / len(df) * 100, 2) for col in df.columns]
    })
    
    # Sort by missing count descending
    missing_info = missing_info.sort_values('Missing Count', ascending=False)
    
    # Format the output
    for _, row in missing_info.iterrows():
        missing_bar = '█' * int(row['Missing %'] / 5)  # Visual bar for missing percentage
        print(f"{row['Column'][:30]:<30} "
              f"| Type: {str(row['Data Type']):<10} "
              f"| Missing: {int(row['Missing Count']):>6} "
              f"| {row['Missing %']:>5}% {missing_bar}")
    
    # Section 3: Unique Values Analysis (for object/category columns)
    print("\n3. UNIQUE VALUES IN CATEGORICAL COLUMNS")
    print("-" * 40)
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    
    if len(categorical_cols) > 0:
        unique_info = pd.DataFrame({
            'Column': categorical_cols,
            'Unique Values': [df[col].nunique() for col in categorical_cols],
            'Sample Values': [str(list(df[col].dropna().unique()[:5])) for col in categorical_cols]
        })
        unique_info = unique_info.sort_values('Unique Values', ascending=False)
        
        for _, row in unique_info.iterrows():
            print(f"{row['Column'][:30]:<30} "
                  f"| Unique: {row['Unique Values']:<5} "
                  f"| Samples: {row['Sample Values'][:50]}")
    else:
        print("No categorical columns found")
    
    # Section 4: Numeric Columns Statistics
    print("\n4. NUMERIC COLUMNS SUMMARY")
    print("-" * 40)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        numeric_stats = df[numeric_cols].describe().T
        numeric_stats['missing'] = df[numeric_cols].isnull().sum()
        print(numeric_stats[['count', 'missing', 'mean', 'min', 'max']])
    else:
        print("No numeric columns found")
    
    return missing_info

def show_value_counts(df, column_name, top_n=10):
    """
    Display value counts for a specific column with formatting
    """
    print("\n" + "=" * 80)
    print(f"VALUE COUNTS FOR COLUMN: '{column_name}'")
    print("=" * 80)
    
    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in DataFrame")
        return
    
    # Get value counts
    counts = df[column_name].value_counts()
    
    # Get value counts with percentages
    counts_pct = df[column_name].value_counts(normalize=True) * 100
    
    # Create a combined DataFrame
    value_counts_df = pd.DataFrame({
        'Count': counts,
        'Percentage': counts_pct,
        'Cumulative %': counts_pct.cumsum()
    })
    
    # Show top N or all if less than top_n
    display_count = min(top_n, len(counts))
    
    print(f"\nTop {display_count} values (out of {len(counts)} unique values):")
    print("-" * 60)
    
    for i, (value, count) in enumerate(counts.head(display_count).items(), 1):
        percentage = counts_pct[value]
        bar = '█' * int(percentage / 2)  # Visual bar (max 50 chars)
        
        # Truncate value if too long
        value_str = str(value)[:30]
        
        print(f"{i:2}. {value_str:<30} "
              f"| Count: {count:<8} "
              f"| {percentage:>5.1f}% {bar}")
    
    # Show missing values if any
    missing_count = df[column_name].isnull().sum()
    if missing_count > 0:
        missing_pct = (missing_count / len(df)) * 100
        print(f"\nMissing values: {missing_count} ({missing_pct:.1f}%)")
    
    return value_counts_df

In [6]:
missing_info = analyze_dataframe(df)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 224 rows × 40 columns
Total cells: 8960
Memory usage: 0.43 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
2024                           | Type: object     | Missing:      8 |  3.57% 
2020                           | Type: object     | Missing:      7 |  3.12% 
2021                           | Type: object     | Missing:      7 |  3.12% 
2022                           | Type: object     | Missing:      7 |  3.12% 
2023                           | Type: object     | Missing:      7 |  3.12% 
2014                           | Type: object     | Missing:      6 |  2.68% 
2011                           | Type: object     | Missing:      6 |  2.68% 
2018                           | Type: object     | Missing:      6 |  2.68% 
2015                           | Type: object     | Missing:      6 |  2.68% 
2016                           | Type: object     | Missing:      6

Step 2: Replace '..' with pandas na, as they are missing values

In [7]:
df = df.replace('..', pd.NA)

In [8]:
df.head()

,country_code,country_name,1987,1988,1989,1990,1991,1992,1993,1994,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,AFG,Afghanistan,L,L,L,L,L,L,L,L,...,L,L,L,L,L,L,L,L,L,L
1,ALB,Albania,<NA>,<NA>,<NA>,LM,LM,LM,L,L,...,UM,UM,UM,UM,UM,UM,UM,UM,UM,UM
2,DZA,Algeria,UM,UM,LM,LM,LM,LM,LM,LM,...,UM,UM,UM,UM,LM,LM,LM,LM,UM,UM
3,ASM,American Samoa,H,H,H,UM,UM,UM,UM,UM,...,UM,UM,UM,UM,UM,UM,UM,H,H,H
4,AND,Andorra,<NA>,<NA>,<NA>,H,H,H,H,H,...,H,H,H,H,H,H,H,H,H,H


Step 3: Melt dataframe

Explanation: At first, each year was a separate column. But we want each year to be a separate row.

In [9]:
# Melt the dataframe: convert years from columns to rows
df_melted = pd.melt(
    df, 
    id_vars=['country_code', 'country_name'],  # Keep these as identifiers
    var_name='year',                            # Column names become year values
    value_name='classification'                 # The values become classification
)

# Sort for better readability (optional)
df_melted = df_melted.sort_values(['country_name', 'year']).reset_index(drop=True)

# Check the result
print(df_melted.shape)  # (8512, 4)
print(df_melted.head())

(8512, 4)
  country_code country_name  year classification
0          AFG  Afghanistan  1987              L
1          AFG  Afghanistan  1988              L
2          AFG  Afghanistan  1989              L
3          AFG  Afghanistan  1990              L
4          AFG  Afghanistan  1991              L


Check

In [10]:
analyze_dataframe(df_melted)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 8512 rows × 4 columns
Total cells: 34048
Memory usage: 1.75 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
classification                 | Type: object     | Missing:    635 |  7.46% █
country_code                   | Type: object     | Missing:      0 |   0.0% 
country_name                   | Type: object     | Missing:      0 |   0.0% 
year                           | Type: object     | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
country_code                   | Unique: 224   | Samples: ['AFG', 'ALB', 'DZA', 'ASM', 'AND']
country_name                   | Unique: 224   | Samples: ['Afghanistan', 'Albania', 'Algeria', 'American Sa
year                           | Unique: 38    | Samples: ['1987', '1988', '1989', '1990', '1991']
classification                 | Unique: 5     | Samples: ['L', 'LM', '

,Column,Data Type,Non-Null Count,Missing Count,Missing %
3,classification,object,7877,635,7.46
0,country_code,object,8512,0,0.00
1,country_name,object,8512,0,0.00
2,year,object,8512,0,0.00


I see 'LM*' as a unique value in 'classification'

This is what the codebook says about that special value:

"* At this time, there were Yemen, PDR (L) and Yemen, Arab Rep. (LM); combined they would have been LM."

Let's check which rows have that classification

In [11]:
mask = df_melted['classification'] == 'LM*'
print(df_melted[mask])

     country_code country_name  year classification
8360          YEM  Yemen, Rep.  1987            LM*
8361          YEM  Yemen, Rep.  1988            LM*


Only two instances. 

As the asterisk only was for explanation for why the classification is noted as LM. We can safely remove the asterisk.

Step 4: Remove asterisks making 'LM*' into 'LM'

In [12]:
# Remove all asterisks from the column
df_melted['classification'] = df_melted['classification'].str.replace('*', '', regex=False)

Check

In [13]:
analyze_dataframe(df_melted)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 8512 rows × 4 columns
Total cells: 34048
Memory usage: 1.75 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
classification                 | Type: object     | Missing:    635 |  7.46% █
country_code                   | Type: object     | Missing:      0 |   0.0% 
country_name                   | Type: object     | Missing:      0 |   0.0% 
year                           | Type: object     | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
country_code                   | Unique: 224   | Samples: ['AFG', 'ALB', 'DZA', 'ASM', 'AND']
country_name                   | Unique: 224   | Samples: ['Afghanistan', 'Albania', 'Algeria', 'American Sa
year                           | Unique: 38    | Samples: ['1987', '1988', '1989', '1990', '1991']
classification                 | Unique: 4     | Samples: ['L', 'LM', '

,Column,Data Type,Non-Null Count,Missing Count,Missing %
3,classification,object,7877,635,7.46
0,country_code,object,8512,0,0.00
1,country_name,object,8512,0,0.00
2,year,object,8512,0,0.00


Ok good

Step 5: Save it to csv

In [14]:
df_melted.to_csv('../../data/processed/unmapped_wb_income_classification.csv', index=False)

Step 6: Missingness analysis

In [16]:
import pandas as pd
import numpy as np

# Assuming your DataFrame is called 'df_melted'

print("=" * 60)
print("MISSING DATA REPORT")
print("=" * 60)

# Overall statistics
total_rows = len(df_melted)
total_missing = df_melted.isnull().sum().sum()
missing_pct = (total_missing / (total_rows * len(df_melted.columns))) * 100

print(f"\nDataset: {total_rows:,} rows, {len(df_melted.columns)} columns")
print(f"Missing values: {total_missing:,} ({missing_pct:.2f}% of all cells)")

# Missing values by column
print("\nMissing Values by Column:")
missing_by_column = df_melted.isnull().sum()
missing_pct_by_column = (df_melted.isnull().sum() / len(df_melted)) * 100

for col in missing_by_column.index:
    missing_count = missing_by_column[col]
    missing_percent = missing_pct_by_column[col]
    if missing_count > 0:
        print(f"   {col:20s}: {missing_count:5d} missing ({missing_percent:5.2f}%)")
    else:
        print(f"   {col:20s}: {missing_count:5d} missing ({missing_percent:5.2f}%)")

# Focus on classification column
if 'classification' in df_melted.columns:
    missing_class = df_melted['classification'].isnull().sum()
    print(f"\nClassification Column Analysis:")
    print(f"   Missing: {missing_class} out of {total_rows} rows ({missing_class/total_rows*100:.2f}%)")
    
    # Show distribution of existing classifications
    print(f"\n   Classification distribution (non-missing only):")
    class_dist = df_melted['classification'].value_counts()
    for class_type, count in class_dist.items():
        print(f"      {class_type}: {count:,} ({count/len(df_melted)*100:.1f}%)")

# Check which countries/years have missing classifications
if 'classification' in df_melted.columns:
    missing_rows = df_melted[df_melted['classification'].isnull()]
    if len(missing_rows) > 0:
        print(f"\nCountries with most missing classifications:")
        country_missing = missing_rows['country_name'].value_counts().head(5)
        for country, count in country_missing.items():
            print(f"   {country[:35]:35s}: {count}")
        
        print(f"\nYears with most missing classifications:")
        year_missing = missing_rows['year'].value_counts().head(5)
        for year, count in year_missing.items():
            print(f"   {year}: {count}")

MISSING DATA REPORT

Dataset: 8,512 rows, 4 columns
Missing values: 635 (1.87% of all cells)

Missing Values by Column:
   country_code        :     0 missing ( 0.00%)
   country_name        :     0 missing ( 0.00%)
   year                :     0 missing ( 0.00%)
   classification      :   635 missing ( 7.46%)

Classification Column Analysis:
   Missing: 635 out of 8512 rows (7.46%)

   Classification distribution (non-missing only):
      H: 2,367 (27.8%)
      LM: 2,093 (24.6%)
      L: 1,755 (20.6%)
      UM: 1,662 (19.5%)

Countries with most missing classifications:
   USSR (former)                      : 37
   Czechoslovakia (former)            : 36
   Yugoslavia (former)                : 33
   British Virgin Islands             : 28
   Nauru                              : 28

Years with most missing classifications:
   1987: 58
   1988: 57
   1989: 54
   1990: 44
   1991: 27


Step 7: Drop missingness

In [17]:
df_melted = df_melted.dropna()

In [18]:
analyze_dataframe(df_melted)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 7877 rows × 4 columns
Total cells: 31508
Memory usage: 1.67 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
country_code                   | Type: object     | Missing:      0 |   0.0% 
country_name                   | Type: object     | Missing:      0 |   0.0% 
year                           | Type: object     | Missing:      0 |   0.0% 
classification                 | Type: object     | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
country_code                   | Unique: 224   | Samples: ['AFG', 'ALB', 'DZA', 'ASM', 'AND']
country_name                   | Unique: 224   | Samples: ['Afghanistan', 'Albania', 'Algeria', 'American Sa
year                           | Unique: 38    | Samples: ['1987', '1988', '1989', '1990', '1991']
classification                 | Unique: 4     | Samples: ['L', 'LM', 'U

,Column,Data Type,Non-Null Count,Missing Count,Missing %
0,country_code,object,7877,0,0.0
1,country_name,object,7877,0,0.0
2,year,object,7877,0,0.0
3,classification,object,7877,0,0.0


Step 8: Save it to csv before heading to next steps

In [20]:
df_melted.to_csv('../../data/raw/v3_unmapped_imf_trade.csv', index=False)

Step 9: Import country_utils_extended module

In [1]:
import sys
import pandas as pd

# sys.path.append("C:\\Users\\vanes\\2024 SCHOOL UVA VANESSA YAN\\YEAR 2\\SEM4\\GP\\Violent-Offenders-GPV---CSSci-\\src")
sys.path.append("../../src")

import importlib
import country_utils_extended as cs
importlib.reload(cs)


<module 'country_utils_extended' from 'c:\\Users\\vanes\\2024 SCHOOL UVA VANESSA YAN\\YEAR 2\\SEM4\\GP\\Violent-Offenders-GPV---CSSci-\\notebooks\\raw-data-exploration\\../../src\\country_utils_extended.py'>

In [2]:
df_melted = pd.read_csv('../../data/raw/v3_unmapped_imf_trade.csv')

Step 9.5: Replace misformatted names

In [3]:
# Directly replace the problem names in your dataframe
df_melted['country_name'] = df_melted['country_name'].replace({
    'Channel Islands': 'Channel Islands',
    'Czechoslovakia (former)': 'Czechoslovakia',
    "Cï¿½te d'Ivoire": "Côte d'Ivoire",
    'Netherlands Antilles (former)': 'Netherlands Antilles',
    'Serbia and Montenegro (former)': 'Serbia and Montenegro',
    'St. Martin (French part)': 'Saint Martin',
    'Sï¿½o Tomï¿½ and Prï¿½ncipe': 'Sao Tome and Principe',
    'Tï¿½rkiye': 'Turkiye',
    'Curaï¿½ao': 'Curaçao'
})

Step 10: Use get_iso3 function from country_utils_extended

In [4]:
df_melted['country_iso3'] = df_melted['country_name'].apply(cs.get_iso3)

Step 11: Use check_coverage function from country_utils_extended

In [5]:
cs.check_coverage(df_melted, 'country_iso3', 'country_name', 'country name')


=== country name ===
Total rows with country_name: 7877
Matched to ISO3: 7877 (100.0%)
Unmatched unique values: 0


array([], dtype=object)

Step 12: Drop country_code that was in the original dataset, because we use iso3 now.

In [7]:
df_melted = df_melted.drop('country_code', axis=1)

Step 12: Save to csv in processed folder. The file is done.

In [8]:
df_melted.to_csv('../../data/processed/v3_imf_trade.csv', index=False)